# Weakly Supervised 3D Disease Localization — Google Colab

BB annotasyonları (2D, kesit bazlı) + TotalSegmentator organ maskeleri →  
**Hastalığın 3D lokalizasyonu** (weakly supervised, pseudo-label değil)

## Yöntem
1. TotalSegmentator **sadece inference** → 6 anatomik organ maskesi
2. BB annotasyonlu kesitlerde: `BB_bölgesi ∩ organ_maskesi`
3. Boundary Slice z-aralığındaki diğer kesitlerde: `organ_maskesi`
4. Dışında: background (0)
5. Çıktı: çok-sınıflı 3D NIfTI  (0=bg, 1..6=hastalık)

## Google Drive Klasör Yapısı
```
MyDrive/abdomen/
    Bilgi.xlsx
    Egitim Verisi/
    Yarisma Veri Seti/
    abdomen_project/
        src/   scripts/
        outputs/splits/  <- manifest.csv burada
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f'GPU: {name} ({mem} GB)')
else:
    print('GPU yok — Runtime > T4 GPU secin.')

## [1] Kullanıcı Ayarları — yalnızca bu hücreyi düzenleyin

In [ ]:
from pathlib import Path

DRIVE_ABDOMEN   = Path('/content/drive/MyDrive/abdomen')
DRIVE_PROJECT   = DRIVE_ABDOMEN / 'abdomen_project'
DRIVE_SPLITS    = DRIVE_PROJECT / 'outputs' / 'splits'
DRIVE_TRAIN_DIR = DRIVE_ABDOMEN / 'Eğitim Verisi.zip'
DRIVE_TEST_DIR  = DRIVE_ABDOMEN / 'Yarisma Veri Seti'
DRIVE_RESULTS   = DRIVE_PROJECT / 'outputs' / 'weak_disease_masks'

TOTALSEG_FAST = True   # T4 16GB icin onerilir
LIMIT         = None   # Debug: kac vaka (None=hepsi)

for p in [DRIVE_ABDOMEN, DRIVE_PROJECT, DRIVE_SPLITS, DRIVE_TRAIN_DIR]:
    print(f'  [{"OK" if p.exists() else "EKSIK"}] {p}')

## [2] Paket Kurulumu

In [ ]:
%%capture
!pip install -q TotalSegmentator SimpleITK pydicom pandas openpyxl tqdm scikit-learn

In [ ]:
import SimpleITK as sitk, pydicom
print(f'SimpleITK : {sitk.Version()}')
print(f'pydicom   : {pydicom.__version__}')
!TotalSegmentator --version

## [3] Proje Kodunu Colab Diskine Kopyala

In [ ]:
import shutil, sys
COLAB_PROJECT = Path('/content/abdomen_project')
if COLAB_PROJECT.exists():
    shutil.rmtree(COLAB_PROJECT)
shutil.copytree(str(DRIVE_PROJECT), str(COLAB_PROJECT))
if str(COLAB_PROJECT) not in sys.path:
    sys.path.insert(0, str(COLAB_PROJECT))
print(f'Proje kopyalandi: {COLAB_PROJECT}')

## [4] Ortam Değişkenlerini Ayarla

In [ ]:
import os
COLAB_SEG_OUT = Path('/content/weak_disease_masks')
COLAB_SEG_OUT.mkdir(parents=True, exist_ok=True)

os.environ['ABDOMEN_PROJECT_ROOT'] = str(COLAB_PROJECT)
os.environ['ABDOMEN_DATA_ROOT']    = str(DRIVE_ABDOMEN)
os.environ['ABDOMEN_TRAIN_DIR']    = str(DRIVE_TRAIN_DIR)
os.environ['ABDOMEN_TEST_DIR']     = str(DRIVE_TEST_DIR)
os.environ['ABDOMEN_BILGI_XLSX']   = str(DRIVE_ABDOMEN / 'Bilgi.xlsx')
os.environ['ABDOMEN_SPLIT_DIR']    = str(DRIVE_SPLITS)
os.environ['ABDOMEN_SEG_DATA_DIR'] = str(COLAB_SEG_OUT)
os.environ['ABDOMEN_OUT_DIR']      = str(COLAB_PROJECT / 'outputs')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

for k, v in sorted(os.environ.items()):
    if k.startswith('ABDOMEN_') or k == 'CUDA_VISIBLE_DEVICES':
        print(f'{k:30s} = {v}')

## [5] Manifest DICOM Yollarını Yeniden Eşle

In [ ]:
import shutil
import pandas as pd

manifest_src = DRIVE_SPLITS / 'manifest.csv'
assert manifest_src.exists(), f'manifest.csv bulunamadi: {manifest_src}'

df     = pd.read_csv(manifest_src)
sample = str(df['dicom_path'].iloc[0])
print(f'Orjinal yol: {sample}')

COLAB_SPLITS = Path('/content/splits_colab')
COLAB_SPLITS.mkdir(exist_ok=True)

if not (sample.startswith('/content/drive') or sample.startswith(str(DRIVE_ABDOMEN))):
    drive_parent = str(DRIVE_ABDOMEN.parent)
    def _remap(p):
        p = str(p)
        return (drive_parent + '/' + p[p.index('abdomen'):]) if 'abdomen' in p else p
    df['dicom_path'] = df['dicom_path'].apply(_remap)
    print(f'Yeniden eslendi: {df["dicom_path"].iloc[0]}')

df.to_csv(COLAB_SPLITS / 'manifest.csv', index=False)
for f in DRIVE_SPLITS.glob('*.csv'):
    if f.name != 'manifest.csv':
        shutil.copy(f, COLAB_SPLITS / f.name)

os.environ['ABDOMEN_SPLIT_DIR'] = str(COLAB_SPLITS)
bb_count = (df['bboxes'].fillna('') != '').sum()
print(f'Toplam satir: {len(df):,}  |  BB annotasyonlu: {bb_count:,}')

## [6] 06b_weak_seg.py — Weakly Supervised 3D Hastalık Maskesi

```
TotalSegmentator (inference)
    -> 6 organ maskesi (3D)
    x  BB annotasyonlari (2D)
    =  3D hastalik maskesi  (0=bg, 1=kolesistit, ..., 6=divertikulit)
```

> **Sure:** ~3-8 dk/vaka (T4, fast mod). 735 egitim vakasi ~ 40-100 saat toplam.  
> Debug icin `LIMIT = 5` kullanin.
>
> **Not:** Apendiks TotalSegmentator'da ayri sinif degildir; kolon maskesinden yaklasik alinir.

In [ ]:
from src.weak_seg import generate_weak_masks

generate_weak_masks(
    limit=LIMIT,
    out_dir=COLAB_SEG_OUT,
    totalseg_fast=TOTALSEG_FAST,
)

## [7] Sonuçları Google Drive'a Kaydet

In [ ]:
import shutil
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
masks = list(COLAB_SEG_OUT.glob('*_disease.nii.gz'))
for p in masks:
    dst = DRIVE_RESULTS / p.name
    if not dst.exists():
        shutil.copy(p, dst)
stats_src = COLAB_SEG_OUT / 'stats.json'
if stats_src.exists():
    shutil.copy(stats_src, DRIVE_RESULTS / 'stats.json')
saved = list(DRIVE_RESULTS.glob('*_disease.nii.gz'))
print(f'Kaydedildi: {DRIVE_RESULTS}')
print(f'Toplam maske: {len(saved)}')

## [8] İstatistikler & Görselleştirme

In [ ]:
import json
import matplotlib.pyplot as plt

with open(COLAB_SEG_OUT / 'stats.json') as f:
    stats = json.load(f)

classes = list(stats['voxel_counts'].keys())
counts  = list(stats['voxel_counts'].values())
print(f'Islenen vaka: {stats["processed_cases"]}')
for cls, n in zip(classes, counts):
    print(f'  {cls:35s}: {n:>12,}')

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
bars = ax.barh(classes, counts, color=colors, alpha=0.85)
ax.bar_label(bars, labels=[f'{n:,}' for n in counts], padding=4, fontsize=9)
ax.set_xlabel('Toplam voxel sayisi')
ax.set_title('Weakly Supervised 3D Hastalik Maskesi — Sinif Dagilimi')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from src.config import SUPER_CLASSES

COLORS = [None,'#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
masks  = sorted(COLAB_SEG_OUT.glob('*_disease.nii.gz'))

if not masks:
    print('Gorsellestirilecek maske yok.')
else:
    mask_path = masks[len(masks) // 2]
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path)))
    dis_slices = np.where(arr.max(axis=(1, 2)) > 0)[0]

    if len(dis_slices) == 0:
        print(f'{mask_path.stem}: hastalık maskesi bos.')
    else:
        mid_z   = dis_slices[len(dis_slices) // 2]
        sl      = arr[mid_z]
        overlay = np.zeros((*sl.shape, 4), dtype=np.float32)
        patches = []
        for cid in range(1, 7):
            region = sl == cid
            if not region.any():
                continue
            rgba = mcolors.to_rgba(COLORS[cid])
            overlay[region] = (*rgba[:3], 0.7)
            patches.append(mpatches.Patch(color=COLORS[cid],
                           label=SUPER_CLASSES[cid - 1].replace('_', ' ')))

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow((sl == 0).astype(float), cmap='gray', alpha=0.4)
        ax.imshow(overlay)
        if patches:
            ax.legend(handles=patches, loc='upper right', fontsize=8)
        ax.set_title(f'{mask_path.stem}  |  z={mid_z}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()